In [17]:
from cresowlve.utils import read_json

en_bench = read_json("../experiments/data/task/chgk_en_benchmark.json")
ru_bench = read_json("../experiments/data/task/chgk_ru_benchmark.json")
reasoning_type_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
factual_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]
creative_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]
en_bench["data"] = [s for s in en_bench["data"] if s["id"] in creative_ids]
ru_bench["data"] = [s for s in ru_bench["data"] if s["id"] in creative_ids]

In [5]:
len(en_bench["data"]), len(ru_bench["data"])

(2061, 2061)

In [4]:
from collections import Counter

difficulties = Counter([s["difficulty_id"] for s in en_bench["data"]])
difficulties

Counter({2: 453, 3: 436, 1: 417, 5: 384, 4: 371})

In [6]:
comments = {i: 0 for i in range(1, 6)}

for sample in en_bench["data"]:
    if sample["comment"] and sample["comment"] != "None":
        comments[sample["difficulty_id"]] += 1

comments, sum(comments.values())

({1: 375, 2: 426, 3: 413, 4: 355, 5: 359}, 1928)

In [12]:
num_en_question_tokens = {i: [] for i in range(1, 6)}
num_en_answer_tokens = {i: [] for i in range(1, 6)}

for sample in en_bench["data"]:
    num_en_question_tokens[sample["difficulty_id"]].append(len(sample["question"].split()))
    num_en_answer_tokens[sample["difficulty_id"]].append(len(sample["answer"].split()))

num_ru_question_tokens = {i: [] for i in range(1, 6)}
num_ru_answer_tokens = {i: [] for i in range(1, 6)}

for sample in ru_bench["data"]:
    num_ru_question_tokens[sample["difficulty_id"]].append(len(sample["question"].split()))
    num_ru_answer_tokens[sample["difficulty_id"]].append(len(sample["answer"].split()))

import numpy as np

latex_template = "{diff} & ${num_samples}$ & ${num_comments}$ & ${en_q}\\pm{en_q_std}$ / ${ru_q}\\pm{ru_q_std}$ & ${en_a}\\pm{en_a_std}$ / ${ru_a}\\pm{ru_a_std}$ \\\\"
diff_name_mapping = {
    1: "Very Simple (1)",
    2: "Simple (2)",
    3: "Medium (3)",
    4: "Hard (4)",
    5: "Very Hard (5)",
}

for i in range(1, 6):
    num_samples = difficulties[i]
    num_comments = comments[i]
    en_q_mean = np.mean(num_en_question_tokens[i])
    en_q_std = np.std(num_en_question_tokens[i])
    ru_q_mean = np.mean(num_ru_question_tokens[i])
    ru_q_std = np.std(num_ru_question_tokens[i])
    en_a_mean = np.mean(num_en_answer_tokens[i])
    en_a_std = np.std(num_en_answer_tokens[i])
    ru_a_mean = np.mean(num_ru_answer_tokens[i])
    ru_a_std = np.std(num_ru_answer_tokens[i])

    print(
        latex_template.format(
            diff=diff_name_mapping[i],
            num_samples=num_samples,
            num_comments=num_comments,
            en_q=f"{en_q_mean:.2f}",
            en_q_std=f"{en_q_std:.2f}",
            ru_q=f"{ru_q_mean:.2f}",
            ru_q_std=f"{ru_q_std:.2f}",
            en_a=f"{en_a_mean:.2f}",
            en_a_std=f"{en_a_std:.2f}",
            ru_a=f"{ru_a_mean:.2f}",
            ru_a_std=f"{ru_a_std:.2f}",
        )
    )

Very Simple (1) & $417$ & $375$ & $37.79\pm12.84$ / $29.65\pm9.90$ & $2.12\pm1.51$ / $1.82\pm1.09$ \\
Simple (2) & $453$ & $426$ & $38.64\pm12.84$ / $30.79\pm10.13$ & $2.20\pm1.73$ / $1.87\pm1.30$ \\
Medium (3) & $436$ & $413$ & $38.11\pm12.13$ / $30.34\pm10.08$ & $2.06\pm1.38$ / $1.87\pm1.10$ \\
Hard (4) & $371$ & $355$ & $38.61\pm13.43$ / $30.60\pm10.35$ & $2.13\pm1.88$ / $1.85\pm1.39$ \\
Very Hard (5) & $384$ & $359$ & $37.92\pm13.05$ / $30.27\pm10.45$ & $2.12\pm1.46$ / $1.89\pm1.22$ \\


In [16]:
# total mean and std
all_en_question_tokens = [len(sample["question"].split()) for sample in en_bench["data"]]
all_en_answer_tokens = [len(sample["answer"].split()) for sample in en_bench["data"]]
all_ru_question_tokens = [len(sample["question"].split()) for sample in ru_bench["data"]]
all_ru_answer_tokens = [len(sample["answer"].split()) for sample in ru_bench["data"]]

total_latex_template = "\\textbf{{{diff}}} & $\\mathbf{{{num_samples}}}$ & $\\mathbf{{{num_comments}}}$ & $\\mathbf{{{en_q}\\pm{en_q_std}}}$ / $\\mathbf{{{ru_q}\\pm{ru_q_std}}}$ & $\\mathbf{{{en_a}\\pm{en_a_std}}}$ / $\\mathbf{{{ru_a}\\pm{ru_a_std}}}$ \\\\"

print(total_latex_template.format(
    diff="Total",
    num_samples=len(en_bench["data"]),
    num_comments=sum(comments.values()),
    en_q=f"{np.mean(all_en_question_tokens):.2f}",
    en_q_std=f"{np.std(all_en_question_tokens):.2f}",
    ru_q=f"{np.mean(all_ru_question_tokens):.2f}",
    ru_q_std=f"{np.std(all_ru_question_tokens):.2f}",
    en_a=f"{np.mean(all_en_answer_tokens):.2f}",
    en_a_std=f"{np.std(all_en_answer_tokens):.2f}",
    ru_a=f"{np.mean(all_ru_answer_tokens):.2f}",
    ru_a_std=f"{np.std(all_ru_answer_tokens):.2f}",
))

\textbf{Total} & $\mathbf{2061}$ & $\mathbf{1928}$ & $\mathbf{38.22\pm12.84}$ / $\mathbf{30.33\pm10.18}$ & $\mathbf{2.13\pm1.60}$ / $\mathbf{1.86\pm1.22}$ \\


In [18]:
human_scores = [sample["result_score"]/sample["result_total"] for sample in en_bench["data"] if sample["id"] in creative_ids]
print(f"Average human score: {np.mean(human_scores):.2f} ± {np.std(human_scores):.2f}")

Average human score: 0.40 ± 0.25
